# V3.2 Counterfactual Branch Credit for CAOSD-RCPO
## Interim Meeting Report

**Meeting focus:** why V3.2 changes branch-level credit assignment, how the counterfactual reward and cost are constructed, and what the current controlled experiments show.

> Status: interim checkpoint snapshot from six 13-hour-budget runs. The runs share the V2.6 Experiment 2 market, seed, constraints, and optimization foundation. Results should be treated as diagnostic evidence rather than a final multi-seed conclusion.

## 1. Starting Point: V2.6

V2.6 combines two complementary mechanisms:

- **CAOSD Simplex Decomposition:** enforces the two allocation constraints by construction.
- **RCPO relative-current-drawdown control:** uses a global cost critic and Lagrange multiplier to discourage excessive drawdown relative to the constrained-neutral benchmark.
- **Autoregressive branch policy:** four CAOSD branches are produced sequentially with either Gaussian logits or Dirichlet branch weights.
- **Standalone reward / global cost credit:** each branch receives a standalone portfolio return signal, while all branches share the final portfolio cost advantage.

The unresolved issue is structural credit assignment. The final portfolio reward and drawdown emerge only after all branch actions are recombined. A standalone branch portfolio is not the same object as that branch's marginal contribution to the final CAOSD portfolio.

## 2. V3.2 Research Question

**Can branch learning become more informative if each branch is trained from the change it causes in the final portfolio, rather than from an isolated branch portfolio or an identical global signal?**

V3.2 adapts counterfactual credit assignment from difference rewards and COMA-style reasoning:

1. Keep the realized actions of all other branches fixed.
2. Replace one branch with its neutral within-branch allocation.
3. Re-run the complete CAOSD composition.
4. Compare the actual and counterfactual portfolio outcomes.

This asks a branch-specific question: **What changed because branch $i$ chose its realized action instead of a neutral action?**

## 3. Counterfactual Construction

Let the realized joint branch action be

$$a_t=(a_{1,t},a_{2,t},a_{3,t},a_{4,t}), \qquad w_t=\operatorname{CAOSD}(a_t).$$

For branch $i$, construct an intervention

$$a_t^{-i}=(a_{1,t},\ldots,a_{i,t}^{neutral},\ldots,a_{4,t}),$$

and recompute the feasible portfolio

$$w_t^{-i}=\operatorname{CAOSD}(a_t^{-i}).$$

The neutral action is zero logits for Gaussian policies and a uniform within-branch allocation for Dirichlet policies. The complete CAOSD map is rerun, so both the actual and counterfactual portfolios satisfy the hard allocation constraints.

The downstream branch actions are held at their realized values. This is an **open-loop, fixed-downstream counterfactual**, not a new policy rollout and not a second market simulation.

## 4. Stateful Counterfactual Reward and Cost

Each branch counterfactual maintains a lightweight shadow path on the same realized market returns:

- previous counterfactual portfolio weights;
- turnover and transaction costs;
- portfolio wealth and running peak;
- current and maximum drawdown.

The counterfactual reward therefore includes its own path-dependent transaction cost:

$$r_{i,t}^{cf}=\log\left(1+w_{i,t,risky}^{cf\top}R_t-\kappa\lVert w_{i,t}^{cf}-w_{i,t-1}^{cf}\rVert_1\right).$$

The branch reward contribution is

$$\Delta r_{i,t}=r_t^{actual}-r_{i,t}^{cf}.$$

The counterfactual path uses the same online constrained-neutral drawdown budget as the actual portfolio. Its branch cost contribution is

$$\Delta c_{i,t}=c_t^{actual}-c_{i,t}^{cf}.$$

$\Delta c_{i,t}$ is signed: a positive value means the realized branch made constraint performance worse than its neutral replacement; a negative value means it improved constraint performance.

## 5. Three V3.2 Credit Variants

| Experiment | Branch reward advantage | Branch cost advantage | Interpretation |
|---|---|---|---|
| Counterfactual reward | $A(\Delta r_i)$ | Global actual $A(c)$ | Test whether marginal return credit is sufficient |
| Counterfactual cost | $z_i A(r_i^{standalone})$ | $A(\Delta c_i)$ | Preserve V2.6 reward credit and isolate marginal risk credit |
| Counterfactual reward + cost | $A(\Delta r_i)$ | $A(\Delta c_i)$ | Give both return and risk branch-specific marginal credit |

For RCPO, the branch actor signal is formed as reward advantage minus $\lambda$ times cost advantage. Counterfactual differences already contain the branch's effect through the CAOSD composition, so they are not multiplied by $z_i$ again. The cost-only experiment retains $z_i$ on the standalone reward term because that reward is computed as a fully invested isolated branch portfolio.

## 6. Critic and Lambda Design

Each active branch has a branch-specific reward critic and, when counterfactual cost is enabled, a branch-specific cost critic. The critic input combines the shared market representation with the branch counterfactual state:

$$[\text{shared features}, w_i^{cf}, turnover_i^{cf}, relative\_wealth_i^{cf}, current\_DD_i^{cf}, max\_DD_i^{cf}, drawdown\_gap_i^{cf}, episode\_progress].$$

One safeguard is essential:

$$\lambda \leftarrow \max(0,\lambda+\eta(\bar c_{actual}-\alpha)).$$

The global Lagrange multiplier is updated **only from the actual final portfolio cost and dynamic alpha**. Signed branch counterfactual costs never update $\lambda$. This prevents a negative marginal cost from being mistaken for globally feasible portfolio behavior.

## 7. Controlled Experiment Design

| Item | Setting |
|---|---|
| Market | Experiment 2 synthetic market: 8 risky assets + cash |
| Training diversity | 8 deterministic train markets |
| Episode length | 252 trading steps |
| Hard constraints | Same two CAOSD allocation constraints as V2.6 |
| RCPO constraint | Relative current drawdown versus constrained-neutral benchmark |
| Benchmark margin / floor | 0.90 / 0.05 |
| Seed | 0, matching V2.6 |
| Architectures | Autoregressive Gaussian and autoregressive Dirichlet |
| V3.2 budget | 11,400 updates, estimated from the 13-hour compute window |
| Test evaluation | 20 common continuation markets, seed offset 20,000 |
| Checkpoints | Best validation return and best validation feasible candidate |

The V2.6 curves are included only in the validation-history figure and are truncated at the longest V3.2 update. Their final checkpoints occurred later (Gaussian at 19,400; Dirichlet at 51,600), so those final models are not used as direct 11,400-update outcome comparisons.

## 8. Validation Learning History

![Validation score history](../evaluation/section9_simplex_v3.2_policy_comparison/section9_validation_score_history.png)

**Reading the figure**

- V2.6 Gaussian and Dirichlet are context lines only and are truncated to the V3.2 horizon.
- Counterfactual reward Gaussian shows the clearest sustained validation improvement in the current window.
- Several Dirichlet variants reach an early high score and then fluctuate or decline, indicating that stability remains unresolved.
- A high validation score alone does not establish drawdown feasibility on unseen branches.

## 9. Best-Return Checkpoint Results

| V3.2 policy | Selected update | Relative wealth vs baseline | Win rate | Mean max drawdown | Test feasible branches |
|---|---:|---:|---:|---:|---:|
| CF Reward Gaussian | 11,000 | **+4.52%** | 70% | 19.19% | 10% |
| CF Cost Gaussian | 10,600 | +4.37% | 75% | **17.33%** | **35%** |
| CF Reward+Cost Gaussian | 10,200 | +4.18% | 70% | 18.42% | 15% |
| CF Reward Dirichlet | 1,400 | **+4.57%** | 75% | 17.76% | 25% |
| CF Cost Dirichlet | 2,000 | +3.81% | 60% | 17.82% | 30% |
| CF Reward+Cost Dirichlet | 4,800 | +4.51% | 70% | 17.66% | 30% |

All six best-return snapshots beat the constrained-neutral baseline on mean relative wealth. However, none generalizes the drawdown constraint across most test branches. The strongest return and strongest test feasibility are achieved by different variants.

![Best-return cumulative comparison](../evaluation/section9_simplex_v3.2_policy_comparison/section9_cumulative_return_comparison.png)

![Best-return drawdown comparison](../evaluation/section9_simplex_v3.2_policy_comparison/section9_max_drawdown_comparison.png)

## 10. Best-Feasible Checkpoint Results

A best-feasible checkpoint is selected on validation data by prioritizing feasible branch rate and then validation return. It is **not guaranteed** to remain feasible on unseen test branches.

| V3.2 policy | Selected update | Relative wealth vs baseline | Mean max drawdown | Test feasible branches |
|---|---:|---:|---:|---:|
| CF Reward Gaussian | 7,800 | **+4.38%** | 18.78% | 10% |
| CF Cost Gaussian | 10,600 | **+4.37%** | 17.33% | 35% |
| CF Reward+Cost Gaussian | 5,800 | +3.81% | 17.85% | 25% |
| CF Reward Dirichlet | 8,800 | +3.58% | 17.79% | 25% |
| CF Cost Dirichlet | 200 | +0.92% | **16.74%** | **40%** |
| CF Reward+Cost Dirichlet | 800 | +4.10% | 17.51% | 30% |

The counterfactual-cost Dirichlet checkpoint is closest to the baseline drawdown (16.70%) and has the highest feasible-branch rate, but sacrifices most of the return improvement. Gaussian counterfactual cost currently provides the most balanced return/feasibility trade-off.

![Best-feasible cumulative comparison](../evaluation/section9_simplex_v3.2_best_feasible_policy_comparison/section9_cumulative_return_comparison.png)

![Best-feasible drawdown comparison](../evaluation/section9_simplex_v3.2_best_feasible_policy_comparison/section9_max_drawdown_comparison.png)

## 11. Main Interpretation

### Positive evidence

- Every current best-return variant has positive mean relative wealth on the common 20-market test set.
- Counterfactual reward can produce a clear learning trend, especially for Gaussian branches.
- Counterfactual cost changes the risk/return profile: the Gaussian cost-only model has lower drawdown and higher test feasibility than the other Gaussian variants.
- Hard allocation feasibility remains exact because all interventions pass through CAOSD.

### Negative evidence

- Reward+cost counterfactual credit does not consistently dominate either single-component intervention.
- Dirichlet best-return checkpoints often occur very early, suggesting unstable later learning rather than reliable convergence.
- Validation-based feasible selection transfers poorly: test feasible rates remain only 10%-40%.
- The experiment currently uses one policy seed, so apparent differences may reflect initialization variance.

The current evidence supports counterfactual credit as a useful diagnostic and potentially useful learning signal, but not yet as a solved replacement for V2.6 branch credit.

## 12. Important Evaluation Correction

An audit of the comparison utility found that RCPO feasibility was initially detected from whether a plot label began with `RCPO`. V3.2 labels begin with `CF`, so the first summary incorrectly reported 100% feasibility by checking only hard allocation constraints.

The utility now identifies RCPO from checkpoint metadata. The corrected feasible rates compare each branch's actual drawdown constraint cost against its dynamic alpha. This correction changes the interpretation substantially:

- hard allocation feasibility remains 100%;
- drawdown feasibility is only 10%-40% across the current test branches;
- therefore the main unresolved issue is risk-constraint generalization, not simplex feasibility.

This correction affects reporting only; it does not change any trained policy or checkpoint.

## 13. Limitations and Next Experiments

1. **Complete equal-budget training.** Compare all variants at the same update horizon rather than relying only on each run's early best checkpoint.
2. **Use multiple policy seeds.** At least three seeds are needed before ranking Gaussian versus Dirichlet or reward versus cost credit.
3. **Audit counterfactual critic quality.** Track explained variance, target variance, $\Delta r_i$/$\Delta c_i$ scale, sign balance, and correlation with realized branch actions.
4. **Improve constraint selection.** Report validation and test feasible-branch rates separately and consider a minimum validation feasible-rate threshold before optimizing return.
5. **Investigate Dirichlet stability.** Examine concentration saturation, entropy, KL, and whether early high-return checkpoints result from transient exploration.
6. **Test closed-loop sensitivity later.** The current open-loop intervention fixes downstream actions. A later ablation could regenerate downstream autoregressive actions after replacing branch $i$, but that changes more than one action and is computationally less controlled.
7. **Preserve V2.6 as the control.** The purpose of V3.2 is to isolate branch-credit effects, not to change the market, allocation constraints, lambda rule, or benchmark simultaneously.

## 14. Discussion Questions

- Is fixed-neutral replacement the most meaningful counterfactual, or should the baseline marginalize over the branch policy as in COMA?
- Should branch cost credit remain signed, or should only harmful positive marginal cost affect the actor?
- Is open-loop fixed-downstream conditioning sufficiently faithful for an autoregressive CAOSD policy?
- Should feasible-checkpoint selection prioritize a minimum feasible-branch rate before return?
- Does the current result justify a multi-seed V3.2 study, or should critic diagnostics be improved first?

## 15. One-Minute Summary

V2.6 gives each simplex branch a standalone return signal and a shared final-portfolio drawdown cost. V3.2 instead measures a branch's marginal contribution by replacing that branch with a neutral allocation, keeping the other realized branch actions fixed, and recomputing the full feasible CAOSD portfolio. I test counterfactual reward, counterfactual cost, and both together for Gaussian and Dirichlet policies, while the global lambda still depends only on the actual portfolio cost. Current interim results show positive return signals for all six variants, with approximately 3.8%-4.6% mean relative wealth for best-return checkpoints. However, corrected evaluation shows only 10%-40% test drawdown-feasible branches. Counterfactual cost improves the risk/return balance in some cases, especially Gaussian, but the method has not yet solved constraint generalization or Dirichlet instability.